# Notebook de Feature Engineering

En este notebook trabajamos con `dataset_modelo_proveedor_v2_candidates_synthetic.csv` para hacer **feature engineering** y evaluar KNN en un problema de clasificación binaria (`target_elegido`).

Cada fila representa un par **(evento de compra, proveedor candidato)**, por lo que el objetivo es estimar la probabilidad de que ese candidato sea el proveedor finalmente elegido.

Usamos validación temporal y comparamos contra un baseline `Dummy` para comprobar si las nuevas variables aportan señal real.


## Librerías

In [ ]:
# Librerías

import pandas as pd
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sys
from pathlib import Path

# Sube de notebooks/ a la raíz del repo

sys.path.append(str(Path.cwd().parent))

# Importamos funciones

import src.ml.functions as fc

%matplotlib inline 
%load_ext autoreload
%autoreload 2

## Cargamos dataset

In [ ]:
df = fc.load_data("dataset_modelo_proveedor_v2_candidates_synthetic.csv")

## EDA

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.shape

In [ ]:
df.isna().sum()

## Subset de datos para entrenamiento

In [ ]:
df_model = fc.df_model_knn_feature(df)

In [ ]:
train, test, _ = fc.split_temporal_feature(df_model)

In [ ]:
feature_cols_num, feature_cols_cat, target_col = fc.get_feature_columns_v2()

In [ ]:
train_num = train[feature_cols_num].apply(pd.to_numeric, errors="coerce")
corr = train_num.corr(numeric_only=True)

display(corr)

high_corr = (
    corr.where(~np.eye(corr.shape[0], dtype=bool))
        .stack()
        .reset_index()
)
high_corr.columns = ["var1", "var2", "corr"]
high_corr = high_corr[high_corr["corr"].abs() >= 0.9].sort_values("corr", ascending=False)
display(high_corr)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlación (train, variables numéricas)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
#feature_cols_num = [
 #   "coste_min_dia_proveedor",
  #  "rank_coste_dia_producto",
   # "terminales_cubiertos",
    #"candidatos_evento_count",
#    "coste_max_evento",
 #   "spread_coste_evento",
  #  "delta_vs_min_evento",
   # "ratio_vs_min_evento",
#    "litros_evento",
 #   "dia_semana",
  #  "mes",
   # "fin_mes"
#]
# Ajuste por colinealidad (train):
# - quitamos `observaciones_oferta` por alta correlación con `terminales_cubiertos` (~0.995)
# - quitamos `coste_min_evento` por alta correlación con `coste_min_dia_proveedor` (~0.955)

# Dejo la celda por trazabilidad pero no se ejecutará dado que empeoró los resultados.

In [ ]:
X_train, X_test, y_train, y_test = fc.dummificar_train_test(train, test, feature_cols_num, feature_cols_cat)

In [ ]:
chosen_k = 21
knn_acc = fc.entrenar_knn(X_train, X_test, y_train, y_test, chosen_k)
print(f"KNN Accuracy (k={chosen_k}): {knn_acc:.4f}")


In [ ]:
# Celda reservada (sin uso).
# En Day 02 evitamos sweep/tuning para no duplicar el trabajo de Day 04.
pass


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

# 1) Baseline Dummy (referencia)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("=== DUMMY ===")
print("accuracy:", round(accuracy_score(y_test, y_pred_dummy), 4))
print("balanced_accuracy:", round(balanced_accuracy_score(y_test, y_pred_dummy), 4))
print("f1_clase_1:", round(f1_score(y_test, y_pred_dummy, pos_label=1, zero_division=0), 4))
print(classification_report(y_test, y_pred_dummy, digits=4, zero_division=0))

# 2) KNN con k fijo definido arriba
knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=chosen_k))
])
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print(f"
=== KNN (k={chosen_k}) ===")
print("accuracy:", round(accuracy_score(y_test, y_pred_knn), 4))
print("balanced_accuracy:", round(balanced_accuracy_score(y_test, y_pred_knn), 4))
print("f1_clase_1:", round(f1_score(y_test, y_pred_knn, pos_label=1, zero_division=0), 4))
print(classification_report(y_test, y_pred_knn, digits=4, zero_division=0))


In [ ]:
import pandas as pd

# 1) Entrenar modelo final con k fijo de esta fase (sin tuning)
best_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=chosen_k))
])
best_model.fit(X_train, y_train)

# 2) Probabilidad de clase positiva (ser elegido)
proba_test = best_model.predict_proba(X_test)
idx_pos = list(best_model.named_steps["knn"].classes_).index(1)
proba_pos = proba_test[:, idx_pos]

# 3) Recuperar test base con event_id + proveedor_candidato
test_eval = test.copy().reset_index(drop=True)
test_eval["score_elegido"] = proba_pos
test_eval["target_elegido"] = y_test.values

# 4) Ranking por evento
test_eval = test_eval.sort_values(["event_id", "score_elegido"], ascending=[True, False])
test_eval["rank_pred"] = test_eval.groupby("event_id").cumcount() + 1

# 5) Métricas Top-1 / Top-2
top1_hit = (
    test_eval[test_eval["rank_pred"] == 1]
    .groupby("event_id")["target_elegido"]
    .max()
    .mean()
)

top2_hit = (
    test_eval[test_eval["rank_pred"] <= 2]
    .groupby("event_id")["target_elegido"]
    .max()
    .mean()
)

print("Top-1 hit rate:", round(top1_hit, 4))
print("Top-2 hit rate:", round(top2_hit, 4))

# 6) Muestra de recomendaciones
cols_show = [
    "event_id", "fecha_evento", "producto_canonico",
    "proveedor_candidato", "score_elegido", "rank_pred", "target_elegido"
]

print("\n=== Ejemplo Top-2 por evento (primeros 10 eventos) ===")
sample_events = test_eval["event_id"].drop_duplicates().head(10)
display(
    test_eval[test_eval["event_id"].isin(sample_events) & (test_eval["rank_pred"] <= 2)][cols_show]
    .sort_values(["event_id", "rank_pred"])
)

# (Opcional) guardar reporte para revisión
# test_eval.to_csv("data/synthetic/inference_outputs/day02_topk_eval_sample.csv", index=False)


## CONCLUSIONES (Day 02 · Feature Engineering sobre V2)

- El `feature engineering` sobre `dataset_modelo_proveedor_v2_candidates_synthetic.csv` mejora claramente el rendimiento de KNN respecto al Day 01.
- En **accuracy** global, KNN queda **muy cerca** del baseline `Dummy` (o lo supera levemente según `k`), por lo que no hay una ventaja contundente si miramos solo esa métrica.
- Esto era esperable por el **desbalance de clases**: `Dummy` (clase mayoritaria) puede obtener una accuracy alta sin aprender la lógica de decisión.
- Para este problema, las métricas más útiles son `balanced_accuracy`, `f1` de la clase positiva (`target_elegido=1`) y métricas de negocio por evento (`Top-1` / `Top-2 hit`).
- Conclusión operativa: KNN + FE es un buen paso intermedio, pero no suficiente como modelo final; sirve como baseline diagnóstico antes de modelos más robustos (LogReg / ensembles).
